In [ ]:
!pip install -q transformers sentencepiece accelerate pandas
!pip install -q scikit-learn
from google.colab import files
import pandas as pd

files.upload()

df = pd.read_csv("xed_dev.tsv", sep="\t")

print("Number of examples:", len(df))
print(df[["english_text", "es_text", "emotion_labels"]].head())

def create_zero_shot_prompt(sentence):
    return f"""Identify the emotions in this sentence.

Possible emotions: anger, anticipation, disgust, fear, joy, sadness, surprise, trust.

Return only a JSON array of the emotions expressed in the sentence.
Do not explain your answer.

Sentence: {sentence}
Answer:"""


import json

EMOTIONS = [
    "anger", "anticipation", "disgust", "fear",
    "joy", "sadness", "surprise", "trust"
]

def parse_prediction(output):
    try:
        labels = json.loads(output.strip())

        if not isinstance(labels, list):
            return [], True

        if not all(isinstance(x, str) for x in labels):
            return [], True

        labels = [x.strip().lower() for x in labels]

        if not all(x in EMOTIONS for x in labels):
            return [], True

        return list(dict.fromkeys(labels)), False

    except (json.JSONDecodeError, TypeError):
        return [], True



Saving xed_dev.tsv to xed_dev (1).tsv
Number of examples: 954
                                        english_text  \
0                             I'll take care of him.   
1  Hey, you know there was a guy in the game that...   
2                  I don't want any lesson, Shooter.   
3                        What happened to your coat?   
4                        - What was the movie about?   

                                             es_text         emotion_labels  
0                             Yo me encargaré de él.       ["joy", "trust"]  
1  ¿Sabes que había un tipo en el juego que me ac...  ["anger", "surprise"]  
2                      No quiero lecciones, Shooter.              ["anger"]  
3                          ¿Qué le pasó a tu abrigo?           ["surprise"]  
4                  - ¿De qué se trataba la película?       ["anticipation"]  


In [5]:
from google.colab import files
import pandas as pd
from pathlib import Path

if not Path("/content/xed_dev.tsv").exists():
    files.upload()  # Select xed_dev.tsv

df = pd.read_csv("/content/xed_dev.tsv", sep="\t")
df = df.reset_index(drop=True)

print("Dataset loaded:", len(df))

Saving xed_dev.tsv to xed_dev.tsv
Dataset loaded: 954


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype="auto"
)

device = "cuda" if torch.cuda.is_available() else "cpu"

model = model.to(device)
model.eval()

print("Model loaded:", model_name)
print("Device:", device)

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model loaded: Qwen/Qwen2.5-1.5B-Instruct
Device: cpu


In [6]:
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")

save_path = Path(
    "/content/drive/MyDrive/qwen_zero_shot_spanish_full_dev.csv"
)

# Resume a previous run if the file exists
if save_path.exists():
    saved_df = pd.read_csv(save_path)

    qwen_full_spanish = saved_df.to_dict("records")

    completed = set(saved_df["row_index"].astype(int))

else:
    qwen_full_spanish = []
    completed = set()

print("Already completed:", len(completed))


for i, row in df.iterrows():

    if i in completed:
        continue

    sentence = row["es_text"]

    messages = [
        {
            "role": "user",
            "content": create_zero_shot_prompt(sentence)
        }
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=60,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    input_length = inputs["input_ids"].shape[-1]

    raw_prediction = tokenizer.decode(
        outputs[0][input_length:],
        skip_special_tokens=True
    ).strip()

    predicted_labels, malformed = parse_prediction(
        raw_prediction
    )

    actual_labels = json.loads(row["emotion_labels"])

    qwen_full_spanish.append({
        "row_index": int(i),
        "sentence": sentence,
        "actual": json.dumps(actual_labels),
        "predicted": json.dumps(predicted_labels),
        "raw_output": raw_prediction,
        "malformed": malformed
    })

    completed.add(int(i))

    if len(qwen_full_spanish) % 25 == 0:
        pd.DataFrame(qwen_full_spanish).to_csv(
            save_path,
            index=False
        )

        print(
            f"Saved {len(qwen_full_spanish)}/{len(df)} examples"
        )

# Final save
pd.DataFrame(qwen_full_spanish).to_csv(
    save_path,
    index=False
)

print("Completed:", len(qwen_full_spanish))
print("Saved to:", save_path)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Already completed: 954
Completed: 954
Saved to: /content/drive/MyDrive/qwen_zero_shot_spanish_full_dev.csv


In [ ]:
import json
import pandas as pd

from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import (
    f1_score,
    precision_score,
    recall_score,
    jaccard_score,
    hamming_loss,
    accuracy_score,
    classification_report
)

EMOTIONS = [
    "anger", "anticipation", "disgust", "fear",
    "joy", "sadness", "surprise", "trust"
]

results_df = pd.read_csv(save_path)

assert len(results_df) == 954
assert results_df["row_index"].nunique() == 954

actual = results_df["actual"].apply(json.loads).tolist()
predicted = results_df["predicted"].apply(json.loads).tolist()

mlb = MultiLabelBinarizer(classes=EMOTIONS)

y_true = mlb.fit_transform(actual)
y_pred = mlb.transform(predicted)

print("QWEN ZERO-SHOT SPANISH — FULL DEVELOPMENT SET")
print("---------------------------------------------")

print("Micro-F1:", f1_score(
    y_true, y_pred, average="micro", zero_division=0
))

print("Macro-F1:", f1_score(
    y_true, y_pred, average="macro", zero_division=0
))

print("Jaccard:", jaccard_score(
    y_true, y_pred, average="samples", zero_division=0
))

print("Micro-Precision:", precision_score(
    y_true, y_pred, average="micro", zero_division=0
))

print("Micro-Recall:", recall_score(
    y_true, y_pred, average="micro", zero_division=0
))

print("Hamming Loss:", hamming_loss(y_true, y_pred))
print("Exact Match:", accuracy_score(y_true, y_pred))

malformed = (
    results_df["malformed"]
    .astype(str)
    .str.lower()
    .eq("true")
)

print("Malformed Rate:", f"{malformed.mean():.2%}")

print("\nPer-emotion results:")
print(classification_report(
    y_true,
    y_pred,
    target_names=EMOTIONS,
    zero_division=0
))

QWEN ZERO-SHOT SPANISH — FULL DEVELOPMENT SET
---------------------------------------------
Micro-F1: 0.31718426501035196
Macro-F1: 0.2818599678554262
Jaccard: 0.2681865828092243
Micro-Precision: 0.358950328022493
Micro-Recall: 0.2841246290801187
Hamming Loss: 0.21606394129979037
Exact Match: 0.16981132075471697
Malformed Rate: 4.51%

Per-emotion results:
              precision    recall  f1-score   support

       anger       0.52      0.17      0.25       248
anticipation       0.33      0.48      0.39       216
     disgust       0.27      0.43      0.33       119
        fear       0.28      0.08      0.13       143
         joy       0.38      0.54      0.44       162
     sadness       0.39      0.36      0.37       138
    surprise       0.21      0.02      0.04       156
       trust       0.47      0.22      0.30       166

   micro avg       0.36      0.28      0.32      1348
   macro avg       0.36      0.29      0.28      1348
weighted avg       0.37      0.28      0.28   

In [ ]:
metrics = {
    "model": "Qwen2.5-1.5B-Instruct",
    "language": "Spanish",
    "method": "Zero-shot",
    "examples": 954,
    "micro_f1": f1_score(y_true, y_pred, average="micro", zero_division=0),
    "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
    "jaccard": jaccard_score(y_true, y_pred, average="samples", zero_division=0),
    "micro_precision": precision_score(y_true, y_pred, average="micro", zero_division=0),
    "micro_recall": recall_score(y_true, y_pred, average="micro", zero_division=0),
    "hamming_loss": hamming_loss(y_true, y_pred),
    "exact_match": accuracy_score(y_true, y_pred),
    "malformed_rate": (
        results_df["malformed"]
        .astype(str)
        .str.lower()
        .eq("true")
        .mean()
    )
}

pd.DataFrame([metrics]).to_csv(
    "/content/drive/MyDrive/qwen_spanish_zero_shot_metrics.csv",
    index=False
)

print("Spanish evaluation metrics saved!")

Spanish evaluation metrics saved!


In [ ]:
from pathlib import Path
import pandas as pd
import json
import torch

english_path = Path(
    "/content/drive/MyDrive/qwen_zero_shot_english_full_dev.csv"
)

# Resume if predictions were previously saved
if english_path.exists():

    saved_df = pd.read_csv(english_path)

    qwen_full_english = saved_df.to_dict("records")

    completed = set(saved_df["row_index"].astype(int))

else:
    qwen_full_english = []
    completed = set()

print("Already completed:", len(completed))

try:
    for i, row in df.iterrows():

        if i in completed:
            continue

        sentence = row["english_text"]

        messages = [
            {
                "role": "user",
                "content": create_zero_shot_prompt(sentence)
            }
        ]

        inputs = tokenizer.apply_chat_template(
            messages,
            tokenize=True,
            add_generation_prompt=True,
            return_tensors="pt",
            return_dict=True
        ).to(model.device)

        with torch.no_grad():

            outputs = model.generate(
                **inputs,
                max_new_tokens=60,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id
            )

        input_length = inputs["input_ids"].shape[-1]

        raw_prediction = tokenizer.decode(
            outputs[0][input_length:],
            skip_special_tokens=True
        ).strip()

        predicted_labels, malformed = parse_prediction(
            raw_prediction
        )

        actual_labels = json.loads(
            row["emotion_labels"]
        )

        qwen_full_english.append({
            "row_index": int(i),
            "sentence": sentence,
            "actual": json.dumps(actual_labels),
            "predicted": json.dumps(predicted_labels),
            "raw_output": raw_prediction,
            "malformed": malformed
        })

        completed.add(int(i))

        # Save every 25 predictions
        if len(qwen_full_english) % 25 == 0:

            pd.DataFrame(qwen_full_english).to_csv(
                english_path,
                index=False
            )

            print(
                f"Saved {len(qwen_full_english)}/{len(df)} examples"
            )

finally:
    if qwen_full_english:
        pd.DataFrame(qwen_full_english).to_csv(
            english_path,
            index=False
        )

print("Completed:", len(qwen_full_english))
print("Saved to:", english_path)

Already completed: 0
Saved 25/954 examples
Saved 50/954 examples
Saved 75/954 examples
Saved 100/954 examples
Saved 125/954 examples
Saved 150/954 examples
Saved 175/954 examples
Saved 200/954 examples
Saved 225/954 examples
Saved 250/954 examples
Saved 275/954 examples
Saved 300/954 examples
Saved 325/954 examples
Saved 350/954 examples
Saved 375/954 examples
Saved 400/954 examples
Saved 425/954 examples
Saved 450/954 examples
Saved 475/954 examples
Saved 500/954 examples
Saved 525/954 examples
Saved 550/954 examples
Saved 575/954 examples
Saved 600/954 examples
Saved 625/954 examples
Saved 650/954 examples
Saved 675/954 examples
Saved 700/954 examples
Saved 725/954 examples
Saved 750/954 examples
Saved 775/954 examples
Saved 800/954 examples
Saved 825/954 examples
Saved 850/954 examples
Saved 875/954 examples
Saved 900/954 examples
Saved 925/954 examples
Saved 950/954 examples
Completed: 954
Saved to: /content/drive/MyDrive/qwen_zero_shot_english_full_dev.csv


In [2]:
import pandas as pd
import json

from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import (
    f1_score,
    precision_score,
    recall_score,
    jaccard_score,
    hamming_loss,
    accuracy_score,
    classification_report
)

EMOTIONS = [
    "anger", "anticipation", "disgust", "fear",
    "joy", "sadness", "surprise", "trust"
]

print("Evaluation libraries loaded!")

Evaluation libraries loaded!


In [7]:
# QWEN ZERO-SHOT ENGLISH — FULL DEVELOPMENT EVALUATION

results_df = pd.read_csv(
    "/content/drive/MyDrive/qwen_zero_shot_english_full_dev.csv"
)

assert len(results_df) == 954
assert results_df["row_index"].nunique() == 954

actual = results_df["actual"].apply(json.loads).tolist()
predicted = results_df["predicted"].apply(json.loads).tolist()

mlb = MultiLabelBinarizer(classes=EMOTIONS)

y_true = mlb.fit_transform(actual)
y_pred = mlb.transform(predicted)

metrics = {
    "model": "Qwen2.5-1.5B-Instruct",
    "language": "English",
    "method": "Zero-shot",
    "examples": 954,
    "micro_f1": f1_score(y_true, y_pred, average="micro", zero_division=0),
    "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
    "jaccard": jaccard_score(y_true, y_pred, average="samples", zero_division=0),
    "micro_precision": precision_score(y_true, y_pred, average="micro", zero_division=0),
    "micro_recall": recall_score(y_true, y_pred, average="micro", zero_division=0),
    "hamming_loss": hamming_loss(y_true, y_pred),
    "exact_match": accuracy_score(y_true, y_pred),
    "malformed_rate": (
        results_df["malformed"]
        .astype(str)
        .str.lower()
        .eq("true")
        .mean()
    )
}

print("QWEN ZERO-SHOT ENGLISH — FULL DEVELOPMENT SET")
print(pd.DataFrame([metrics]).to_string(index=False))

print("\nPer-emotion results:")
print(classification_report(
    y_true,
    y_pred,
    target_names=EMOTIONS,
    zero_division=0
))

pd.DataFrame([metrics]).to_csv(
    "/content/drive/MyDrive/qwen_english_zero_shot_metrics.csv",
    index=False
)

print("English evaluation metrics saved!")

QWEN ZERO-SHOT ENGLISH — FULL DEVELOPMENT SET
                model language    method  examples  micro_f1  macro_f1  jaccard  micro_precision  micro_recall  hamming_loss  exact_match  malformed_rate
Qwen2.5-1.5B-Instruct  English Zero-shot       954   0.34465  0.309472 0.288417         0.409602      0.297478      0.199817      0.19392        0.134172

Per-emotion results:
              precision    recall  f1-score   support

       anger       0.63      0.30      0.41       248
anticipation       0.35      0.52      0.42       216
     disgust       0.28      0.31      0.29       119
        fear       0.37      0.10      0.15       143
         joy       0.43      0.49      0.46       162
     sadness       0.47      0.35      0.40       138
    surprise       0.36      0.06      0.11       156
       trust       0.46      0.16      0.23       166

   micro avg       0.41      0.30      0.34      1348
   macro avg       0.42      0.29      0.31      1348
weighted avg       0.43     